In [ ]:
# for use in ucrits(n) function
def choices(n, k):
    if (n < 0) | (k < 0):
        return []
    if (n == 0) & (k == 0):
        return [0]
    return [2*l for l in choices(n-1, k)] + [2*l + 1 for l in choices(n-1, k-1)]

def pasctomat(pt):
    return ((pt[0] - pt[1])//2, pt[1]//2)
def mattopasc(pt):
    return (2*(pt[0] + pt[1]), 2*pt[1])

def pqbitstoset(p, q, num):
    corners = set([])
    for i in range(p*q):
        if num % 2 == 1:
            corners.add(mattopasc((i//p, i%p)))
        num = num//2
    return corners
# corners variable is a set of 2-tuples, each of which has two even Pascal's triangle coordinates

def bitstoset(n, num):
    return pqbitstoset(n, n, num)

In [ ]:
# goal: findfast(corners) function, which gives the paths (if admits critical) or None (if not)
def plen(path):
    return 1 + path[3] - path[1]

def rightofdot(pair):
    return (pair[0]+1, pair[1]+1)
def downofdot(pair):
    return (pair[0]+1, pair[1])
def leftofbox(pair):
    return (pair[0] - 2, pair[1] - 2)
def upofbox(pair):
    return (pair[0] - 2, pair[1])

def faststepper(corners, donepaths, progress):
    # corners is the set of corners
    # donepaths is the list of completed paths, each path given as a 4-tuple, which we will edit
    # progress is a 4-tuple giving the start point of a path in progress (or null) and which point we just checked
    if progress[2] == progress[3]:
        # we need to start a row
        if progress[2] == 0:
            # very beginning state
            currpoint = (1, 0)
        else:
            currpoint = (progress[2] + 2, 0)
            if progress[0] != 0:
                # we need to finish the path we were in the middle of
                if plen(progress) % 3 == 1:
                    return None
                donepaths.append(progress)
                progress = (0, 0, 0, 0)
    else:
        currpoint = (progress[2], progress[3] + 1)
    if currpoint[1] % 2 == 1:
        # we're to the left of a grid box
        if (leftofbox(rightofdot(currpoint)) not in corners) & (rightofdot(currpoint) in corners):
            # then we get a dot!
            if progress[0] == 0:
                # we start a new path
                return (currpoint[0], currpoint[1], currpoint[0], currpoint[1])
            else:
                # connect our current point to the old path
                return (progress[0], progress[1], currpoint[0], currpoint[1])
        else:
            # no dot!
            if progress[0] == 0:
                return (0, 0, currpoint[0], currpoint[1])
            else:
                # need to save the completed path
                if plen(progress) % 3 == 1:
                    return None
                donepaths.append(progress)
                return (0, 0, currpoint[0], currpoint[1])
    else:
        # we're to the top of a grid box
        if (upofbox(downofdot(currpoint)) not in corners) & (downofdot(currpoint) in corners):
            # then we get a dot!
            if progress[0] == 0:
                # we start a new path
                return (currpoint[0], currpoint[1], currpoint[0], currpoint[1])
            else:
                # need to check whether we should connect or if we can fit a 2x2
                if leftofbox(upofbox(downofdot(currpoint))) in corners:
                    # connect
                    return (progress[0], progress[1], currpoint[0], currpoint[1])
                else:
                    # finish old path, start new path
                    if plen(progress) % 3 == 1:
                        return None
                    donepaths.append(progress)
                    return (currpoint[0], currpoint[1], currpoint[0], currpoint[1])
        else:
            # no dot!
            if progress[0] == 0:
                return (0, 0, currpoint[0], currpoint[1])
            else:
                # need to save the completed path
                if plen(progress) % 3 == 1:
                    return None
                donepaths.append(progress)
                return (0, 0, currpoint[0], currpoint[1])

def findfast(corners):
    maxdiag = max([c[0] for c in corners])
    progress = (0, 0, 0, 0)
    donepaths = []
    while progress[2] <= maxdiag:
        progress = faststepper(corners, donepaths, progress)
        if progress == None:
            return None
    return donepaths
# donepaths is a list of paths, and each path is a 4-tuple of 2 Pascal's triangle start coords and 2 Pasc. end coords

In [ ]:
def crituconf(corners, paths):
    # takes in an unlabeled set of corners, returns a labeled configuration (labels in order of Pascal coords)
    labcorners = sorted(corners)
    ans = {}
    for i in range(len(labcorners)):
        c = labcorners[i]
        ans[c] = (0, 0, i+1)
    for p in paths:
        start = p[1]
        length = p[3] - p[1] + 1
        for i in range(length):
            if i % 3 == 1:
                # our "path string" should have a 1 at this spot
                pos = start + i
                if pos % 2 == 1:
                    # left of a box
                    boxcode = rightofdot((p[0], pos))
                    ans[boxcode] = (1, (ans[boxcode])[1], (ans[boxcode])[2])
                else:
                    # top of a box
                    boxcode = downofdot((p[0], pos))
                    ans[boxcode] = ((ans[boxcode])[0], 1, (ans[boxcode])[2])
    return ans
# answer is same as variable conf, a dictionary where the keys are Pascal's triangle pairs of coordinates, 
# and the values are a 3-tuple (h, v, L), where h is 1 if width 2, v is 1 if height 2, and L is the label number

def dimconf(conf):
    return sum([(conf[x])[0] + (conf[x])[1] for x in conf])

def ucrits(n):
    ans = [[] for i in range(n)]
    for num in choices(n^2, n):
        corners = bitstoset(n, num)
        paths = findfast(corners)
        if paths != None:
            uconf = crituconf(corners, paths)
            ans[dimconf(uconf)].append(uconf)
    return ans
# answer is a list containing, for each possible dimension, a list of critical confs of that dimension

In [ ]:
ucrits(2)

In [ ]:
# These two functions find the max dimension of a critical cell for various p,q (over all n)
# (it's the table I made for Matt)
def maxcritdim(p, q):
    maxdim = 0
    bestn = set([])
    for num in range(1, 2^(p*q)):
        corners = pqbitstoset(p, q, num)
        paths = findfast(corners)
        if paths != None:
            uconf = crituconf(corners, paths)
            dim = dimconf(uconf)
            if dim == maxdim:
                bestn.add(len(corners))
            elif dim > maxdim:
                maxdim = dim
                bestn = set([len(corners)])
    return [maxdim, bestn]

def table(pmax, qmax):
    for q in range(2, qmax + 1):
        for p in range(2, min(pmax + 1, q+1)):
            print (p, q, maxcritdim(p, q))

In [ ]:
%prun f = table(4, 5)

In [ ]:
threecrits = ucrits(3)
[len(L) for L in threecrits]

In [ ]:
fourcrits = ucrits(4)
[len(L) for L in fourcrits]

In [ ]:
fivecrits = ucrits(5)
[len(L) for L in fivecrits]

In [ ]:
sixcrits = ucrits(6)
[len(L) for L in sixcrits]

In [ ]:
def flattenconf(conf):
    corners = sorted(conf)
    ans = []
    for c in corners:
        ans.append(c[0])
        ans.append(c[1])
    for c in corners:
        code = conf[c]
        ans.append(code[0])
        ans.append(code[1])
    for c in corners:
        code = conf[c]
        ans.append(code[2])
    return tuple(ans)

def permute(fconf, perm):
    n = len(fconf)//5
    oldlabels = fconf[4*n:5*n]
    newlabels = [perm[k-1] for k in oldlabels]
    return fconf[:4*n] + tuple(newlabels)

def remember(fconf, chain, perm, indict):
    newconf = permute(fconf, perm)
    if newconf not in indict:
        newchain = {}
        for key in chain:
            newchain[permute(key, perm)] = chain[key]
        indict[newconf] = newchain

def helpperms(L):
    if len(L) == 0:
        return [[]]
    else:
        ans = []
        for i in range(len(L)):
            newL = list(L[:i])+list(L[i+1:])
            ans = ans + [p+[L[i]] for p in helpperms(newL)]
        return ans
def allperms(n):
    return [tuple(perm) for perm in helpperms(range(1, n+1))]

def bigremember(fconf, chain, permlist, indict):
    for p in permlist:
        remember(fconf, chain, p, indict)

def startconfdict(ucritlist, permlist):
    ans = {}
    for L in ucritlist:
        for conf in L:
            fconf = flattenconf(conf)
            chain = {}
            chain[fconf] = 1
            bigremember(fconf, chain, permlist, ans)
    return ans

In [ ]:
d2 = startconfdict(ucrits(2), allperms(2))
d2

In [ ]:
def pathstepper(corners, donepaths, progress):
    # corners is the set of corners
    # donepaths is the list of completed paths, each path given as a 4-tuple, which we will edit
    # progress is a 4-tuple giving the start point of a path in progress (or null) and which point we just checked
    if progress[2] == progress[3]:
        # we need to start a row
        if progress[2] == 0:
            # very beginning state
            currpoint = (1, 0)
        else:
            currpoint = (progress[2] + 2, 0)
            if progress[0] != 0:
                # we need to finish the path we were in the middle of
                donepaths.append(progress)
                progress = (0, 0, 0, 0)
    else:
        currpoint = (progress[2], progress[3] + 1)
    if currpoint[1] % 2 == 1:
        # we're to the left of a grid box
        if (leftofbox(rightofdot(currpoint)) not in corners) & (rightofdot(currpoint) in corners):
            # then we get a dot!
            if progress[0] == 0:
                # we start a new path
                return (currpoint[0], currpoint[1], currpoint[0], currpoint[1])
            else:
                # connect our current point to the old path
                return (progress[0], progress[1], currpoint[0], currpoint[1])
        else:
            # no dot!
            if progress[0] == 0:
                return (0, 0, currpoint[0], currpoint[1])
            else:
                # need to save the completed path
                donepaths.append(progress)
                return (0, 0, currpoint[0], currpoint[1])
    else:
        # we're to the top of a grid box
        if (upofbox(downofdot(currpoint)) not in corners) & (downofdot(currpoint) in corners):
            # then we get a dot!
            if progress[0] == 0:
                # we start a new path
                return (currpoint[0], currpoint[1], currpoint[0], currpoint[1])
            else:
                # need to check whether we should connect or if we can fit a 2x2
                if leftofbox(upofbox(downofdot(currpoint))) in corners:
                    # connect
                    return (progress[0], progress[1], currpoint[0], currpoint[1])
                else:
                    # finish old path, start new path
                    donepaths.append(progress)
                    return (currpoint[0], currpoint[1], currpoint[0], currpoint[1])
        else:
            # no dot!
            if progress[0] == 0:
                return (0, 0, currpoint[0], currpoint[1])
            else:
                # need to save the completed path
                donepaths.append(progress)
                return (0, 0, currpoint[0], currpoint[1])
            
def findpaths(corners):
    maxdiag = max([c[0] for c in corners])
    progress = (0, 0, 0, 0)
    donepaths = []
    while progress[2] <= maxdiag:
        progress = pathstepper(corners, donepaths, progress)
    return donepaths

def dotcode(conf, dot):
    if dot[1] % 2 == 1:
        # left of box
        return (conf[rightofdot(dot)])[0]
    else:
        # top of box
        return (conf[downofdot(dot)])[1]
def labdotswap(conf, dot):
    if dot[1] % 2 == 1:
        code = conf[rightofdot(dot)]
        conf[rightofdot(dot)] = (1 - code[0], code[1], code[2])
    else:
        code = conf[downofdot(dot)]
        conf[downofdot(dot)] = (code[0], 1 - code[1], code[2])
        
def labmatchconf(conf):
    # only for those that are not critical and are guaranteed to be legal configurations
    corners = set([x for x in conf])
    paths = findpaths(corners)
    for p in paths:
        start = p[1]
        length = plen(p)
        for i in range(length):
            pos = (p[0], start + i)
            if i % 3 == 0:
                if dotcode(conf, pos) == 1:
                    # then match by changing it to 0
                    newconf = conf.copy()
                    labdotswap(newconf, pos)
                    return newconf
                if (dotcode(conf, pos) == 0) & (length == i + 1):
                    # we have 010 ... 010 0, match by changing it to 1
                    newconf = conf.copy()
                    labdotswap(newconf, pos)
                    return newconf
            elif i % 3 == 1:
                if dotcode(conf, pos) == 0:
                    # then match by changing the previous 0 to 1
                    newconf = conf.copy()
                    prevpos = (p[0], start + i - 1)
                    labdotswap(newconf, prevpos)
                    return newconf
    return None

In [ ]:
sampleconf = {(0, 0):(0, 0, 1), (4, 2):(1, 0, 2), (4, 4):(1, 0, 3)}
labmatchconf(sampleconf)

In [ ]:
# These functions are used to keep track of signs.
def precdim(conf):
    # given a conf, returns a dictionary with the same keys, giving for each corner how many 1's we've seen before it
    ans = {}
    count = 0
    for corner in sorted(conf):
        ans[corner] = count
        count = count + (conf[corner])[0] + (conf[corner])[1]
    return ans

def slabboundary(conf):
    # same as unlabeled, but the codes have a third entry for the label, 's' stands for "signed"
    ans = []  
    sign = 1
    confprec = precdim(conf)
    for corner in sorted(conf):
        code = conf[corner]
        if code[0] == 1:
            # horizontal
            newconf = conf.copy()
            newconf[corner] = (0, code[1], code[2])
            ans.append([newconf, sign])
            newconf = conf.copy()
            del newconf[corner]
            newconf[leftofbox(corner)] = (0, code[1], code[2])
            facesign = 1
            if code[1] == 1:
                newconfprec = precdim(newconf)
                facesign = (-1)^(confprec[corner] - newconfprec[leftofbox(corner)]) # number of 1's dragged past
            ans.append([newconf, -sign*facesign])
            sign = -sign
        if code[1] == 1:
            # vertical
            newconf = conf.copy()
            newconf[corner] = (code[0], 0, code[2])
            ans.append([newconf, sign])
            newconf = conf.copy()
            del newconf[corner]
            newconf[upofbox(corner)] = (code[0], 0, code[2])
            facesign = 1
            if code[0] == 1:
                newconfprec = precdim(newconf)
                facesign = (-1)^(confprec[corner] - newconfprec[upofbox(corner)])
            ans.append([newconf, -sign*facesign])
            sign = -sign
    return ans

In [ ]:
slabboundary(sampleconf)

In [ ]:
# True: F2 chain complex (coefficients mod 2, boundary matrices over GF(2), mod-2 Betti).
# False: integer coefficients and rational rank (original). Re-run confdict/bdrydict after changing.
BOUNDARY_MOD2 = True

def substitute(fconf, sbdry):
    # warning: mutates sbdry!
    for c in sbdry:
        if fconf == flattenconf(c[0]):
            sign = c[1]
            sbdry.remove(c)
    if sign == 1:
        # if fconf appeared with a positive sign, we replace by the *negative* of everything else
        out = [[c[0], -c[1]] for c in sbdry]
    else:
        out = sbdry
    if BOUNDARY_MOD2:
        out = [[c[0], int(c[1]) % 2] for c in out if int(c[1]) % 2 != 0]
    return out

def reduceconf(conf, permlist, confdict):
    # warning: we're assuming that indict already contains all the critical configs in all permutations
    fconf = flattenconf(conf)
    if fconf in confdict:
        return confdict[fconf]
    match = labmatchconf(conf)
    if dimconf(match) < dimconf(conf):
        # then our conf reduces to nothing!
        chain = {}
        bigremember(fconf, chain, permlist, confdict)
        return chain
    else:
        # need to match up, take boundary, substitute
        prechain = substitute(fconf, slabboundary(match))
        chain = {}
        for c in prechain:
            # add the reduced version of that conf, with the appropriate coefficient
            redc = reduceconf(c[0], permlist, confdict)
            for basefconf in redc:
                nv = chain.get(basefconf, 0) + c[1] * redc[basefconf]
                if BOUNDARY_MOD2:
                    nv = int(nv) % 2
                if nv == 0:
                    chain.pop(basefconf, None)
                else:
                    chain[basefconf] = nv
        for basefconf in sorted(list(chain.keys())):
            if chain[basefconf] == 0:
                del chain[basefconf]
        bigremember(fconf, chain, permlist, confdict)
        return chain


In [ ]:
sampconf = {(2, 0):(0, 0, 2), (2, 2):(0, 0, 1)}
print(reduceconf(sampconf, allperms(2), d2))
print(d2)

In [ ]:
def redboundary(crconf, permlist, confdict, bdrydict):
    # only for critical inputs!
    fconf = flattenconf(crconf)
    if fconf in bdrydict:
        return bdrydict[fconf]
    if dimconf(crconf) == 0:
        chain = {}
        bigremember(fconf, chain, permlist, bdrydict)
        return chain
    prechain = slabboundary(crconf)
    chain = {}
    for c in prechain:
        # add the reduced version of that conf, with the appropriate coefficient
        redc = reduceconf(c[0], permlist, confdict)
        for basefconf in redc:
            nv = chain.get(basefconf, 0) + c[1] * redc[basefconf]
            if BOUNDARY_MOD2:
                nv = int(nv) % 2
            if nv == 0:
                chain.pop(basefconf, None)
            else:
                chain[basefconf] = nv
    for basefconf in sorted(list(chain.keys())):
        if chain[basefconf] == 0:
            del chain[basefconf]
    bigremember(fconf, chain, permlist, bdrydict)
    return chain

def makebdrydict(ucritlist, permlist, confdict):
    # confdict must already be initialized with crit configs
    ans = {}
    for L in ucritlist:
        for crconf in L:
            redboundary(crconf, permlist, confdict, ans)
    return ans


In [ ]:
twocrits = ucrits(2)
myperms = allperms(2)
twoconfdict = startconfdict(twocrits, myperms)
twobdrydict = makebdrydict(twocrits, myperms, twoconfdict)
print(twobdrydict)

In [ ]:
def inrect(conf, p, q):
    maxrow = max([pasctomat(c)[0] for c in conf])
    maxcol = max([pasctomat(c)[1] for c in conf])
    return (maxrow < p) & (maxcol < q)

def fcrits(ucritlist, permlist, p, q):
    ans = []
    for i in range(len(ucritlist)):
        dimifconfs = []
        for conf in ucritlist[i]:
            if inrect(conf, p, q):
                fconf = flattenconf(conf)
                for perm in permlist:
                    dimifconfs.append(permute(fconf, perm))
        ans.append(dimifconfs)
    return ans

In [ ]:
fcrits(twocrits, myperms, 2, 1)

In [ ]:
from sage.all import GF

def ranks(fcritlist, bdrydict):
    ans = []
    for i in range(len(fcritlist)):
        if i == 0:
            ans.append(0)
        else:
            upcells = fcritlist[i]
            downcells = fcritlist[i-1]
            if BOUNDARY_MOD2:
                adj = matrix(GF(2), len(downcells), len(upcells))
            else:
                adj = matrix(len(downcells), len(upcells))
            for j in range(len(upcells)):
                fconf = upcells[j]
                chain = bdrydict[fconf]
                for basefconf in chain:
                    v = chain[basefconf]
                    if BOUNDARY_MOD2:
                        v = int(v) % 2
                    adj[downcells.index(basefconf), j] = v
            ans.append(rank(adj))
    return ans

def crcounts(fcritlist):
    return [len(L) for L in fcritlist]
def bettis(ranklist, crcountlist):
    ranklist.append(0)
    return [crcountlist[j] - ranklist[j] - ranklist[j+1] for j in range(len(crcountlist))]


In [ ]:
# --- How the (ordered) matrices are made ---
# 1. ucrits(n) lists one critical cell per *unlabeled* shape (canonical labels 1..n).
# 2. startconfdict stores each under all n! permutations -> confdict keys = ordered critical cells.
# 3. makebdrydict: for each critical cell, slabboundary gives signed boundary faces; reduceconf
#    expresses each face as a chain of critical cells; result stored under all perms in bdrydict.
# 4. fcrits(ucritlist, permlist, p, q) expands to ALL permutations in the rectangle -> ordered generators.
# 5. ranks(fcritlist, bdrydict) builds the boundary matrix: rows = dim-(i-1) cells, cols = dim-i cells.
#
# --- Unordered configuration space: one generator per unlabeled cell (no n! expansion) ---
# Unlabeled key = flattenconf but without labels: fconf[:4*n] (positions + h,v per corner, no label).

def unlabeled_key(conf):
    """Hashable key for the unlabeled orbit (labels ignored)."""
    f = flattenconf(conf)
    return f[:4 * len(conf)]

def makebdrydict_unordered(ucritlist, bdrydict):
    """Boundary dict for the unordered chain complex. Keys = unlabeled_key; values = chain as dict ukey -> coeff."""
    ans = {}
    for L in ucritlist:
        for conf in L:
            fconf = flattenconf(conf)
            n = len(conf)
            ukey = fconf[:4 * n]
            if ukey in ans:
                continue
            chain_ordered = bdrydict[fconf]
            unlabeled_chain = {}
            for ofconf, coeff in chain_ordered.items():
                u = ofconf[:4 * n]
                if BOUNDARY_MOD2:
                    nv = (unlabeled_chain.get(u, 0) + int(coeff)) % 2
                    if nv == 0:
                        unlabeled_chain.pop(u, None)
                    else:
                        unlabeled_chain[u] = nv
                else:
                    unlabeled_chain[u] = unlabeled_chain.get(u, 0) + coeff
            if not BOUNDARY_MOD2:
                unlabeled_chain = {u: c for u, c in unlabeled_chain.items() if c != 0}
            ans[ukey] = unlabeled_chain
    return ans

def fcrits_unordered(ucritlist, p, q):
    """Critical cells per dimension for unordered complex in p x q rectangle (one per unlabeled cell)."""
    if not ucritlist or not ucritlist[0]:
        return []
    ans = []
    for i in range(len(ucritlist)):
        ans.append([unlabeled_key(conf) for conf in ucritlist[i] if inrect(conf, p, q)])
    return ans


In [ ]:
# Unordered: same ranks/bettis pipeline, but with 1/n! the generators
bdry_unord = makebdrydict_unordered(twocrits, twobdrydict)
funord = fcrits_unordered(twocrits, 2, 2)
runord = ranks(funord, bdry_unord)
cunord = crcounts(funord)
bunord = bettis(runord[:], cunord)
print("n=2 unordered: ranks", runord, "counts", cunord, "Betti", bunord)
print("n=2 ordered:   counts", crcounts(fcrits(twocrits, myperms, 2, 2)), "-> ratio", [a // max(1, b) for a, b in zip(crcounts(fcrits(twocrits, myperms, 2, 2)), cunord)])

In [ ]:
# Build and display the 2x2 unordered boundary matrix (dim 1 -> dim 0), over F2 if BOUNDARY_MOD2
# Requires: twocrits, twobdrydict, unordered helpers; re-run confdict/bdrydict after toggling BOUNDARY_MOD2
bdry_unord = makebdrydict_unordered(twocrits, twobdrydict)
funord = fcrits_unordered(twocrits, 2, 2)
downcells = funord[0]
upcells = funord[1]
if BOUNDARY_MOD2:
    adj = matrix(GF(2), len(downcells), len(upcells))
else:
    adj = matrix(len(downcells), len(upcells))
for j in range(len(upcells)):
    for base_ukey, coeff in bdry_unord[upcells[j]].items():
        if base_ukey in downcells:
            v = int(coeff) % 2 if BOUNDARY_MOD2 else coeff
            adj[downcells.index(base_ukey), j] = v
print("2x2 unordered boundary matrix (rows = dim 0, cols = dim 1; entries in GF(2) when BOUNDARY_MOD2):")
print(adj)
print("Ranks:", [0, rank(adj)])
print("Betti (mod 2 if BOUNDARY_MOD2):", bettis([0, rank(adj)], [len(downcells), len(upcells)]))


In [ ]:
twoflist = fcrits(twocrits, myperms, 2, 2)
tworlist = ranks(twoflist, twobdrydict)
twoclist = crcounts(twoflist)
twoblist = bettis(tworlist, twoclist)
print(tworlist, twoclist, twoblist)

In [ ]:
threeperms = allperms(3)
confdict3 = startconfdict(threecrits, threeperms)
bdrydict3 = makebdrydict(threecrits, threeperms, confdict3)
f3 = fcrits(threecrits, threeperms, 3, 3)
r3 = ranks(f3, bdrydict3)
c3 = crcounts(f3)
b3 = bettis(r3, c3)
print(r3, c3, b3)

In [ ]:
fourperms = allperms(4)
confdict4 = startconfdict(fourcrits, fourperms)
bdrydict4 = makebdrydict(fourcrits, fourperms, confdict4)
f4 = fcrits(fourcrits, fourperms, 4, 4)
r4 = ranks(f4, bdrydict4)
c4 = crcounts(f4)
b4 = bettis(r4, c4)
print(r4, c4, b4)

In [ ]:
# These tables are for the collapsed chain complex.  The r vector gives the ranks of the boundary maps, 
# the c vector gives the dimensions of the chain spaces (this time with the n! factor), and
# the b vector gives the Betti numbers.
rects = [(1, 4), (2, 4), (3, 4), (2, 3), (3, 3)]
for rec in rects:
    f = fcrits(fourcrits, fourperms, rec[0], rec[1])
    r = ranks(f, bdrydict4)
    c = crcounts(f)
    b = bettis(r, c)
    print("p=", rec[0], ", q=", rec[1], ": ", r, c, b)

In [ ]:
fiveperms = allperms(5)
confdict5 = startconfdict(fivecrits, fiveperms)
bdrydict5 = makebdrydict(fivecrits, fiveperms, confdict5)

In [ ]:
rects = [(2, 3), (2, 4), (2, 5), (3, 3), (3, 4), (3, 5), (4, 4), (4, 5), (5, 5)]
for rec in rects:
    f = fcrits(fivecrits, fiveperms, rec[0], rec[1])
    r = ranks(f, bdrydict5)
    c = crcounts(f)
    b = bettis(r, c)
    print("p=", rec[0], ", q=", rec[1], ": ", b)

In [ ]:
sixperms = allperms(6)
confdict6 = startconfdict(sixcrits, sixperms)
bdrydict6 = makebdrydict(sixcrits, sixperms, confdict6)

In [ ]:
rects = [(2, 3), (2, 4), (2, 5)]
for rec in rects:
    f = fcrits(fivecrits, fiveperms, rec[0], rec[1])
    r = ranks(f, bdrydict5)
    c = crcounts(f)
    b = bettis(r, c)
    print("p=", rec[0], ", q=", rec[1], ": ", b)

In [ ]:
%prun f = fcrits(fivecrits, fiveperms, 5, 5)

In [ ]:
%prun r = ranks(f, bdrydict5)

In [ ]:
# These functions are for printing out for human viewing the rectangle representations of the cells in our complex
def flatshowconf(fconf):
    n = len(fconf)//5
    ans = matrix(n, n)
    for i in range(n):
        coords = pasctomat((fconf[2*i], fconf[2*i + 1]))
        shortcode = fconf[2*n+2*i:2*n+2*i+2]
        label = fconf[4*n+i]
        ans[coords[0], coords[1]] = label
        if shortcode[0] == 1:
            ans[coords[0], coords[1] - 1] = label
        if shortcode[1] == 1:
            ans[coords[0] - 1, coords[1]] = label
            if shortcode[0] == 1:
                ans[coords[0] - 1, coords[1] - 1] = label
    return ans

In [ ]:
sampleconf = {(0, 0):(0, 0, 1), (4, 2):(1, 0, 2), (4, 4):(1, 0, 3)}
%display plain
flatshowconf(flattenconf(sampleconf))

In [ ]:
# It's weirdly a pain to print out several matrices next to each other, so the next few boxes deal with that.
L = [matrix([[1, 2], [3, 4]]), matrix([[5, 6], [7, 8]])]
for i in range(2):
    for M in L:
        print(M[i:i+1]),
    print('\n'),

In [ ]:
def helpprmatlist(L):
    if len(L) == 0:
        return
    n = L[0].nrows()
    for i in range(n):
        for M in L:
            print(M[i:i+1]),
        print('\n'),
    print('\n'),

def prmatlist(L, k):
    if len(L) <= k:
        helpprmatlist(L)
    else:
        helpprmatlist(L[:k])
        prmatlist(L[k:], k)
    

In [ ]:
m1 = matrix([[1, 2], [3, 4]])
m2 = matrix([[5, 6], [7, 8]])
L = [m1, m2, m2, m1, m2, m1, m1, m2, m2, m1, m1, m2, m1, m2, m2, m1]
prmatlist(L, 5)

In [ ]:
# This is to print out the boundary map in the collapsed chain complex, so we can look for patterns by human eye.
# (The strings of numbers are the coefficients in the boundary chain of a given critical cell.)
def prbound(n):
    ucritlist = ucrits(n)
    permlist = allperms(n)
    confdict = startconfdict(ucritlist, permlist)
    bdrydict = makebdrydict(ucritlist, permlist, confdict)
    for dim in range(n):
        for conf in ucritlist[dim]:
            fconf = flattenconf(conf)
            print (flatshowconf(fconf), " has the following boundary:")
            bdlist = []
            for fc in bdrydict[fconf]:
                bdlist.append(flatshowconf(fc))
                print((bdrydict[fconf])[fc])
            print ('\n')
            prmatlist(bdlist, 20//n)


In [ ]:
prbound(2)

In [ ]:
prbound(3)

In [ ]:
prbound(4)

In [ ]:
prbound(5)